In [1]:
import torch
import math
from torch.distributions import Categorical, Normal
torch.autograd.set_detect_anomaly(True)

In [2]:
d_logits = torch.tensor([
    [0, -10, 90, 0],
    [100, 0, 0, -100],
    [25, 25, 25, 25]
], dtype=torch.float, requires_grad=True)
def d_mask(d_logits):
    return d_logits.where(torch.tensor([
        [True, False, False, False],
        [True, False, False, True],
        [True, True, True, True]
    ]), -torch.inf)
d_mask(d_logits)

tensor([[   0.,  -inf,  -inf,  -inf],
        [ 100.,  -inf,  -inf, -100.],
        [  25.,   25.,   25.,   25.]], grad_fn=<WhereBackward0>)

In [3]:
log_std = torch.zeros(1, requires_grad=True)
c_logits = torch.tensor([0, 100.5, -10], dtype=torch.float, requires_grad=True)

In [4]:
def d_stats(d_logits):
    dist = Categorical(logits=d_mask(d_logits))
    print("Probs:", dist.probs)
    trans_index = torch.tensor([0, 0, 0])
    log_p = dist.log_prob(trans_index)
    other_logs = dist.logits.scatter(-1, trans_index.unsqueeze(-1), -torch.inf)
    log_1_p = other_logs.where(other_logs.isfinite().any(-1, keepdim=True), math.log(1 / other_logs.size(-1))).logsumexp(-1)
    # log_1_p = other_logs.where(other_logs.isfinite().any(-1, keepdim=True), -torch.inf).logsumexp(-1)
    # log_1_p = other_logs.where(other_logs.isfinite().any(-1, keepdim=True), 0).logsumexp(-1)
    # log_1_p = other_logs.logsumexp(-1)
    return log_p, log_1_p

In [5]:
def c_stats(c_logits, log_std):
    dist = Normal(c_logits, log_std.exp())
    trans_index = torch.tensor([0, 100, 0])
    log_p = (dist.cdf(trans_index + 1) - dist.cdf(trans_index) + 1e-8).log()
    log_1_p = (1 - (dist.cdf(trans_index + 1) - dist.cdf(trans_index)) + 1e-8).log()
    # log_p = dist.log_prob(trans_index)
    # log_1_p = (1 - log_p.exp()).log()
    return log_p, log_1_p

In [6]:
adv = torch.tensor([-1, -1, -1])
def my_loss(log_p, log_1_p, *_):
    return -log_p.where(adv >= 0, log_1_p) * adv.abs()
def pos_loss(log_p, *_):
    return -log_p * adv
def ppo_loss(log_p, _, old_log_p):
    return -(log_p - old_log_p).exp() * adv

In [7]:
steps = 0
old_log_p = c_stats(c_logits, log_std)[0].detach()

In [284]:
loss = my_loss(*c_stats(c_logits, log_std), old_log_p)
print("Loss:", loss)
if c_logits.grad is not None:
    c_logits.grad.zero_()
if log_std.grad is not None:
    log_std.grad.zero_()
loss.mean().backward()
with torch.no_grad():
    c_logits -= c_logits.grad
    log_std -= log_std.grad
steps += 1
steps, c_logits, log_std

Loss: tensor([0.0039, 0.0039, 0.0039], grad_fn=<MulBackward0>)


(277,
 tensor([ -0.1793, 100.5000, -10.0473], requires_grad=True),
 tensor([4.6356], requires_grad=True))